[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milvus-io/bootcamp/blob/master/bootcamp/RAG/search_with_jev/evaluation_with_jev.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/milvus-io/bootcamp/blob/master/bootcamp/RAG/search_with_jev/evaluation_with_jev.ipynb)

# Evaluate search evidence with Jev

Use Jev as a judge after retrieval: assess passage relevance, evidence sufficiency and whether a proposed answer is supported. Compare a few judgments with hand-written labels to illustrate judge validation, not to claim benchmark quality.

## Preparation

Run locally from this directory with `uv sync --python 3.12` and `uv run jupyter lab`, or install the notebook dependencies in Colab:

In [1]:
# In Colab, uncomment this setup cell. Local users should use uv sync --python 3.12.
# %pip install "pymilvus>=2.5,<2.6.10" "milvus-lite>=2.5,<3" "setuptools<71" scikit-learn requests

> In Colab, restart the runtime after installing dependencies if needed.

Set `TYPESAFE_API_KEY` in your environment or enter it privately below. Only the small synthetic examples are sent to TypeSafe. Requests consume API credits.

The helper batches independent questions into one request. IDs map responses back to code; the instructions explicitly identify each field being judged. HTTP failures stop the tutorial rather than produce fabricated scores.

In [2]:
import getpass
import json
import math
import os
import time
import uuid

import requests
from pymilvus import DataType, MilvusClient
from sklearn.feature_extraction.text import TfidfVectorizer

if not os.getenv("TYPESAFE_API_KEY"):
    os.environ["TYPESAFE_API_KEY"] = getpass.getpass("TypeSafe API key: ")

MODEL = os.getenv("JEV_MODEL", "jev-1.13.0")
API_URL = "https://api.typesafe.ai/v1/systemone"
call_log = []

### Call Jev

This helper sends independent questions in one request and validates the returned answers.

In [3]:
def judge(state, questions):
    """Call Jev with bounded retries; stop on invalid or incomplete responses."""
    for attempt in range(3):
        started = time.perf_counter()
        response = requests.post(
            API_URL,
            headers={"Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}"},
            json={"model": MODEL, "state": state, "questions": questions},
            timeout=45,
        )
        if response.status_code in (429, 500, 502, 503, 504) and attempt < 2:
            time.sleep(2**attempt)
            continue
        response.raise_for_status()
        body = response.json()
        answers = body["answers"]
        if set(answers) != set(questions):
            raise ValueError("Jev returned missing or unexpected question IDs")
        for key, question in questions.items():
            answer = answers[key]
            if question["type"] == "noul":
                value = float(answer["noul"])
                if not math.isfinite(value) or not 0 <= value <= 1:
                    raise ValueError("Invalid Noul probability")
            elif answer["choice"] not in question["criteria"]:
                raise ValueError("Unexpected Choice option")
        call_log.append(
            {
                "seconds": round(time.perf_counter() - started, 3),
                "usage": body.get("usage", {}),
                "model": MODEL,
            }
        )
        return answers
    raise RuntimeError("Jev request failed")


def noul(instructions):
    return {
        "type": "noul",
        "instructions": instructions,
        "criteria": {
            "true": "The stated condition is supported by the supplied data.",
            "false": "The condition is unsupported or contradicted.",
        },
    }

## Prepare a small corpus

All names and records below are synthetic teaching examples.

In [4]:
documents = [
    {
        "id": 1,
        "text": "Atlas installation quickstart: install Docker, download the compose file, then run docker compose up. Includes a complete beginner walkthrough.",
        "category": "docs",
        "version": "v2",
    },
    {
        "id": 2,
        "text": "Atlas production installation: configure TLS, backups, health checks and recovery procedures. Intended for experienced operators.",
        "category": "docs",
        "version": "v2",
    },
    {
        "id": 3,
        "text": "Atlas installation announcement: our new release is faster. This announcement contains no installation commands.",
        "category": "news",
        "version": "v2",
    },
    {
        "id": 4,
        "text": "Atlas billing: invoices are available from the billing settings page.",
        "category": "billing",
        "version": "v2",
    },
    {
        "id": 5,
        "text": "Atlas installation v1: use the legacy setup script. This procedure is obsolete for v2.",
        "category": "docs",
        "version": "v1",
    },
    {
        "id": 6,
        "text": "Database integration tests failed because DATABASE_URL used localhost inside a container. Fix: use the compose service hostname db.",
        "category": "memory",
        "version": "v2",
    },
    {
        "id": 7,
        "text": "Database integration tests are run with pytest tests/integration. This note records the command, not a connection failure fix.",
        "category": "memory",
        "version": "v2",
    },
]

## Connect to Milvus

For `MilvusClient`:

- Use a local file such as `./search_with_jev.db` for [Milvus Lite](https://milvus.io/docs/milvus_lite.md).
- Set `MILVUS_URI` to a server endpoint such as `http://localhost:19530` for [Milvus on Docker or Kubernetes](https://milvus.io/docs/quickstart.md).
- For [Zilliz Cloud](https://zilliz.com/cloud), set `MILVUS_URI` to the public endpoint and `MILVUS_TOKEN` to your API key.

Each run uses its own collection name. Cleanup removes only that collection.

In [5]:
client = MilvusClient(
    uri=os.getenv("MILVUS_URI", "./search_with_jev.db"),
    token=os.getenv("MILVUS_TOKEN", ""),
)
collection_name = "jev_demo_" + uuid.uuid4().hex[:12]

## Encode the sample documents

These small examples use TF-IDF lexical vectors to avoid downloading an embedding model or needing another API key. Use the same encoder for indexing and querying; replace it with a dense embedding model for production semantic retrieval.

In [6]:
# Tiny lexical vectors keep this tutorial CPU-only and avoid a second API key.
# Replace this encoder with your production embedding model for semantic retrieval.
encoder = TfidfVectorizer()
vectors = encoder.fit_transform([row["text"] for row in documents]).toarray()

## Create the collection

Define the primary key, vector and text fields explicitly. Additional sample metadata is stored in dynamic fields. The vector index and search both use cosine similarity.

In [7]:
schema = client.create_schema(auto_id=False, enable_dynamic_field=True)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(
    field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=vectors.shape[1]
)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=8192)
index_params = client.prepare_index_params()
index_params.add_index(
    field_name="vector", index_type="AUTOINDEX", metric_type="COSINE"
)
if not client.has_collection(collection_name=collection_name):
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params,
        # consistency_level="Strong",
    )

## Insert the documents

Write the document text, metadata and vectors to Milvus.

In [8]:
client.insert(
    collection_name=collection_name,
    data=[dict(row, vector=vector.tolist()) for row, vector in zip(documents, vectors)],
)

{'insert_count': 7, 'ids': [1, 2, 3, 4, 5, 6, 7], 'cost': 0}

## Retrieve candidates

Return only text and sample metadata, keeping vectors out of the Jev request. Strong consistency makes newly inserted documents searchable immediately.

In [9]:
output_fields = sorted({key for row in documents for key in row if key != "vector"})


def retrieve(query, limit=5, filter_expr=""):
    vector = encoder.transform([query]).toarray()[0]
    if not vector.any():
        return []
    hits = client.search(
        collection_name=collection_name,
        data=[vector.tolist()],
        anns_field="vector",
        limit=limit,
        filter=filter_expr,
        output_fields=output_fields,
        search_params={"metric_type": "COSINE", "params": {}},
        consistency_level="Strong",
    )[0]
    return [
        dict(
            {key: value for key, value in hit["entity"].items() if key != "vector"},
            id=hit["id"],
            retrieval_score=hit["distance"],
        )
        for hit in hits
    ]

## Evaluate retrieved evidence

Keep the proposed answer fixed so this notebook makes no generation-model call. Jev scores are judgments, not ground-truth retrieval metrics.

In [10]:
query = "How do I install Atlas v2?"
candidates = retrieve(query, limit=4, filter_expr='version == "v2"')
proposed_answer = (
    "Install Docker, download the v2 compose file, and run docker compose up."
)
questions = {
    f"relevant_{i}": noul(
        f"Does `candidates[{i}].text` provide concrete installation instructions for `query`?"
    )
    for i in range(len(candidates))
}
questions["sufficient"] = noul(
    "Does `candidates` contain enough evidence to provide basic installation steps for `query`?"
)
questions["supported"] = noul(
    "Is every substantive claim in `proposed_answer` supported by `candidates`? Check support in the supplied text, not truth from outside knowledge."
)
answers = judge(
    {"query": query, "candidates": candidates, "proposed_answer": proposed_answer},
    questions,
)
manual_relevant_ids = {1, 2}
comparisons = [
    (
        row["id"],
        answers[f"relevant_{i}"]["noul"] >= 0.5,
        row["id"] in manual_relevant_ids,
    )
    for i, row in enumerate(candidates)
]
print("ID, judge decision, manual label:", comparisons)
print(
    "Toy agreement:",
    sum(pred == label for _, pred, label in comparisons) / len(comparisons),
)
print("Sufficiency:", answers["sufficient"], "Answer support:", answers["supported"])
# Retrieval recall uses manual labels, not Jev's own scores.
print(
    "Toy retrieval recall:",
    len({row["id"] for row in candidates} & manual_relevant_ids)
    / len(manual_relevant_ids),
)

ID, judge decision, manual label: [(1, True, True), (4, False, False), (2, False, True), (3, False, False)]
Toy agreement: 0.75
Sufficiency: {'type': 'noul', 'noul': 0.96} Answer support: {'type': 'noul', 'noul': 0.92}
Toy retrieval recall: 1.0


## Inspect usage and clean up

The raw usage fields and request duration help inspect this run. They are not a latency benchmark.

In [11]:
print(json.dumps(call_log, indent=2))
client.drop_collection(collection_name=collection_name)
client.close()

[
  {
    "seconds": 0.617,
    "usage": {
      "input_tokens": 1002,
      "output_tokens": 119
    },
    "model": "jev-1.13.0"
  }
]


## Next steps

The thresholds in this example are starting points, not calibrated production defaults. Independent questions share state but do not see each other's answers. See the [Jev primitives](https://docs.typesafe.ai/primitives) and the [cookbook index](https://github.com/milvus-io/bootcamp/blob/master/bootcamp/RAG/search_with_jev/README.md).